# CellAI — Growth Curve Fits (Round 1 + Round 2 + Round 3)

Two fitting approaches compared:
1. **Fitted offset**: `y = y0 + K / (1 + exp(-r*(t-t0)))` — 4 free parameters (K, r, t0, y0)
2. **Fixed offset**: `y = y0_fixed + K / (1 + exp(-r*(t-t0)))` — 3 free parameters (K, r, t0), y0 = first data point

- **Round 1**: 10 wells (3 SD, 3 HB, 3 LBv2, 1 NB control)
- **Round 2**: 13 wells (4 BayBE HB, 2 BayBE SD, 6 Elnora mechanistic, 1 NB control)
- **Round 3**: 11 wells (3 BayBE HB, 2 BayBE SD, 5 Elnora, 1 NB control)

In [ ]:
import csv
import io
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (16, 20)
plt.rcParams['font.size'] = 10

In [ ]:
# Round 1 raw OD600 data
r1_csv_text = """timestamp,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11
2026-03-15T01:53:06.299872+00:00,0.1606,0.1472,0.1561,0.1355,0.1434,0.1479,0.1155,0.1166,0.1202,0.1224
2026-03-15T02:03:00.596871+00:00,0.1668,0.1542,0.1627,0.1437,0.1507,0.1553,0.1181,0.1193,0.1223,0.1264
2026-03-15T02:12:58.695954+00:00,0.1825,0.1659,0.1741,0.1564,0.1639,0.1682,0.1234,0.1253,0.1271,0.1365
2026-03-15T02:22:58.971090+00:00,0.1999,0.18,0.191,0.1748,0.1813,0.1839,0.1315,0.1336,0.1338,0.1461
2026-03-15T03:03:03.722581+00:00,0.3738,0.3443,0.3639,0.343,0.3435,0.3456,0.2132,0.2475,0.2328,0.2677
2026-03-15T03:12:59.940438+00:00,0.4318,0.3959,0.4232,0.4029,0.4017,0.3934,0.2419,0.2842,0.2704,0.285
2026-03-15T03:23:00.022410+00:00,0.4651,0.4295,0.4635,0.4328,0.4395,0.4477,0.2751,0.3309,0.3156,0.3157
2026-03-15T03:33:02.248492+00:00,0.4852,0.4477,0.4882,0.4455,0.4722,0.4846,0.2994,0.3666,0.3491,0.3297
2026-03-15T03:43:00.457277+00:00,0.4924,0.4581,0.502,0.4545,0.4869,0.5028,0.315,0.3921,0.3751,0.3434
2026-03-15T03:53:00.669364+00:00,0.4961,0.4675,0.5134,0.4622,0.491,0.5153,0.3277,0.4098,0.3958,0.3351
2026-03-15T04:06:52.813051+00:00,0.5049,0.5339,0.5952,0.5393,0.5567,0.5875,0.3966,0.479,0.4541,0.3461
2026-03-15T04:13:07.452103+00:00,0.4972,0.4684,0.5245,0.4714,0.4901,0.5229,0.3369,0.4235,0.4134,0.3528
2026-03-15T04:25:14.281076+00:00,0.5001,0.4664,0.5362,0.474,0.4882,0.5223,0.3446,0.4278,0.4203,0.3567"""

r1_conditions = {
    'B2': ('SD-1', 'YE=7, Tryp=13, MOPS=55mM'),
    'B3': ('SD-2', 'YE=10, Tryp=7, MOPS=35mM'),
    'B4': ('SD-3', 'YE=13, Tryp=10, MOPS=45mM'),
    'B5': ('HB-1', 'YE=7, Tryp=13, NaGl=50mM'),
    'B6': ('HB-2', 'YE=10, Tryp=7, NaGl=30mM'),
    'B7': ('HB-3', 'YE=13, Tryp=10, NaGl=40mM'),
    'B8': ('LB-1', 'MOPS=60, KH2PO4=50, Glu=9'),
    'B9': ('LB-2', 'MOPS=80, KH2PO4=30, Glu=6'),
    'B10': ('LB-3', 'MOPS=90, KH2PO4=40, Glu=4'),
    'B11': ('NB-CTRL', 'Novel Bio Cyclone 180uL'),
}

# Round 2 raw OD600 data
r2_csv_text = """timestamp,B2,B3,B4,B5,B6,B7,B8,D2,D3,D4,D5,D6,D7
2026-03-15T07:31:07.634282+00:00,0.1633,0.1638,0.1632,0.1807,0.161,0.1855,0.1433,0.1627,0.1636,0.2687,0.1419,0.2268,0.1958
2026-03-15T08:06:07.436336+00:00,0.2435,0.2548,0.2564,0.2655,0.2591,0.2806,0.2267,0.2453,0.2544,0.3632,0.2154,0.3208,0.2559
2026-03-15T08:16:02.545981+00:00,0.2585,0.2686,0.2705,0.2764,0.2628,0.2863,0.2501,0.2578,0.2623,0.3688,0.2212,0.3224,0.2472
2026-03-15T08:26:02.876070+00:00,0.2907,0.2977,0.2952,0.3076,0.2835,0.3187,0.2731,0.2904,0.294,0.3871,0.2397,0.3346,0.2627
2026-03-15T08:36:01.479898+00:00,0.3737,0.3478,0.3351,0.3447,0.3116,0.3486,0.2956,0.3378,0.3224,0.4148,0.2601,0.3416,0.2772
2026-03-15T08:46:02.200482+00:00,0.4066,0.4077,0.4085,0.3963,0.3857,0.3993,0.3216,0.3966,0.3831,0.4731,0.3159,0.3526,0.295
2026-03-15T08:56:04.390497+00:00,0.4519,0.4528,0.4557,0.4705,0.4292,0.4754,0.33,0.4534,0.4548,0.5323,0.369,0.3717,0.3228
2026-03-15T09:06:02.545025+00:00,0.4946,0.4933,0.4821,0.5178,0.4702,0.5085,0.3334,0.4783,0.4963,0.5603,0.4132,0.3791,0.3461
2026-03-15T09:16:02.712241+00:00,0.5145,0.5314,0.5194,0.542,0.5106,0.5309,0.3331,0.5062,0.5367,0.5962,0.4565,0.3884,0.3802
2026-03-15T09:26:02.933662+00:00,0.5221,0.5519,0.5358,0.5517,0.5407,0.5538,0.3385,0.5183,0.5532,0.644,0.4924,0.3943,0.4185
2026-03-15T09:36:03.104527+00:00,0.5311,0.5678,0.5534,0.5607,0.5635,0.569,0.3477,0.5234,0.5699,0.6828,0.511,0.3993,0.4492
2026-03-15T09:46:01.332394+00:00,0.5367,0.5728,0.5588,0.5713,0.5738,0.5712,0.356,0.5259,0.5969,0.7047,0.519,0.4047,0.4672
2026-03-15T09:56:01.525291+00:00,0.5386,0.571,0.5573,0.582,0.5759,0.5759,0.3663,0.5254,0.599,0.722,0.5177,0.4079,0.4808"""

r2_conditions = {
    'B2': ('HB-1 (BayBE)', 'YE=5.0, Tryp=14.0, NaGl=50mM'),
    'B3': ('HB-2 (BayBE)', 'YE=8.2, Tryp=11.3, NaGl=50mM'),
    'B4': ('HB-3 (BayBE)', 'YE=8.4, Tryp=14.9, NaGl=50mM'),
    'B5': ('SD-1 (BayBE)', 'YE=15.0, Tryp=10.6, MOPS=47mM'),
    'B6': ('HB-4 (BayBE)', 'YE=6.5, Tryp=12.6, NaGl=40mM'),
    'B7': ('SD-3 (BayBE)', 'YE=12.5, Tryp=8.6, MOPS=55mM'),
    'B8': ('NB-CTL', 'Novel Bio Cyclone 180uL'),
    'D2': ('HBD-MaxN (Elnora)', 'HB+Tryp30+YE20+NaGl15'),
    'D3': ('SD-Glut (Elnora)', 'SD+Tryp26+YE14+NaGl10+MOPS10'),
    'D4': ('HBD-Metals (Elnora)', 'HB+Tryp26+YE14+NaGl10+MgSO4+Trace'),
    'D5': ('HBD-Gly (Elnora)', 'HB+Tryp20+YE14+NaGl10+Glycerol36'),
    'D6': ('HBD-FeCit (Elnora)', 'HB+Tryp26+YE14+NaGl10+FeSO4+NaCit'),
    'D7': ('DefGly-Rich (Elnora)', 'DefGly+Tryp26+YE20+MOPS14'),
}

# Round 3 raw OD600 data (preliminary - 9 reads)
r3_csv_text = """timestamp,D8,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11
2026-03-15T18:11:09.434760+00:00,0.1245,0.9239,0.3163,0.24,0.3651,0.3066,0.2664,0.3227,0.2899,0.305,0.308
2026-03-15T18:16:56.767983+00:00,0.1267,0.9138,0.3175,0.2414,0.3679,0.3103,0.2676,0.3238,0.291,0.3033,0.3096
2026-03-15T18:27:12.790381+00:00,0.1345,0.9089,0.3205,0.25,0.3773,0.3267,0.2764,0.3257,0.2965,0.3036,0.3122
2026-03-15T18:37:11.088330+00:00,0.1428,0.8934,0.3251,0.259,0.3901,0.3485,0.2864,0.3316,0.3057,0.3056,0.3154
2026-03-15T18:47:13.543847+00:00,0.157,0.8954,0.3331,0.2679,0.4026,0.3648,0.2945,0.3386,0.315,0.3073,0.3174
2026-03-15T18:57:11.592380+00:00,0.1701,0.895,0.3331,0.2709,0.4048,0.3747,0.296,0.3383,0.3168,0.3089,0.3191
2026-03-15T19:07:10.446928+00:00,0.1906,0.8975,0.3374,0.2746,0.4094,0.382,0.2998,0.3386,0.3198,0.3102,0.3205
2026-03-15T19:17:10.849391+00:00,0.2253,0.9031,0.3398,0.2776,0.4124,0.3874,0.3027,0.3397,0.323,0.306,0.3232
2026-03-15T19:27:08.961675+00:00,0.2481,0.9042,0.3401,0.2793,0.4144,0.3913,0.3046,0.3406,0.3251,0.3057,0.3247"""

r3_conditions = {
    'F2': ('HB-1 (BayBE)', 'YE=12.4, Tryp=7.7, NaGl=34mM'),
    'F3': ('HB-2 (BayBE)', 'YE=7.8, Tryp=6.3, NaGl=26mM'),
    'F4': ('HB-3 (BayBE)', 'YE=5.1, Tryp=6.5, NaGl=33mM'),
    'F5': ('SD-1 (BayBE)', 'YE=7.3, Tryp=13.4, MOPS=36mM'),
    'F6': ('SD-2 (BayBE)', 'YE=5.0, Tryp=15.0, MOPS=73mM'),
    'F7': ('HB-Opt1 (Elnora)', 'HB+Tryp30+YE15+NaGl12'),
    'F8': ('HB-Rich (Elnora)', 'HB+Tryp40+YE30+NaGl10'),
    'F9': ('HB-CarbonMix (Elnora)', 'HB+Tryp25+YE25+Glu10+Glyc10'),
    'F10': ('HB-Phos (Elnora)', 'HB+Tryp30+YE20+KH2PO4=5'),
    'F11': ('SD-CarbonMix (Elnora)', 'SD+Tryp20+YE20+Glu10+Glyc10'),
    'D8': ('NB-CTL', 'Novel Bio Cyclone 180uL'),
}

In [ ]:
def logistic(t, K, r, t0, y0):
    """Logistic with offset: y0 + K / (1 + exp(-r*(t-t0)))"""
    return y0 + K / (1.0 + np.exp(-r * (t - t0)))

def to_unix(ts):
    return datetime.fromisoformat(ts.strip()).timestamp()

def parse_data(csv_text, conditions):
    reader = csv.DictReader(io.StringIO(csv_text))
    wells_data = {}
    for row in reader:
        ts = row['timestamp']
        for well in conditions:
            val = row.get(well, '')
            if val:
                wells_data.setdefault(well, []).append((ts, float(val)))
    return wells_data

def fit_all_fitted_offset(wells_data):
    """4-parameter fit: K, r, t0, y0 all fitted."""
    fit_results = {}
    for well, points in wells_data.items():
        points.sort()
        t0_unix = to_unix(points[0][0])
        times = np.array([(to_unix(ts) - t0_unix) / 3600 for ts, _ in points])
        ods = np.array([od for _, od in points])
        baseline = ods[0]
        K0 = float(np.max(ods) - np.min(ods))
        t0_0 = float(times[np.argmin(np.abs(ods - (np.min(ods) + K0/2)))])
        y0_0 = float(np.min(ods))
        try:
            popt, _ = curve_fit(logistic, times, ods, p0=[K0, 1.0, t0_0, y0_0],
                               bounds=([0, 0, -np.inf, 0], [np.inf, np.inf, np.inf, np.inf]),
                               maxfev=10000)
            K, r, t0, y0 = popt
            y_pred = logistic(times, *popt)
            ss_res = np.sum((ods - y_pred)**2)
            ss_tot = np.sum((ods - np.mean(ods))**2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else None
            fit_results[well] = {
                'times': times, 'ods': ods, 'popt': popt,
                'K': K, 'r': r, 't0': t0, 'y0': y0, 'r2': r2,
                'baseline': baseline, 'biomass_gain': K,
            }
        except Exception as e:
            fit_results[well] = {
                'times': times, 'ods': ods, 'popt': None,
                'K': None, 'r': None, 't0': None, 'y0': None, 'r2': None,
                'baseline': baseline, 'biomass_gain': None, 'error': str(e),
            }
    return fit_results

def fit_all_fixed_offset(wells_data):
    """3-parameter fit: K, r, t0 fitted; y0 fixed to first data point."""
    fit_results = {}
    for well, points in wells_data.items():
        points.sort()
        t0_unix = to_unix(points[0][0])
        times = np.array([(to_unix(ts) - t0_unix) / 3600 for ts, _ in points])
        ods = np.array([od for _, od in points])
        baseline = ods[0]
        K0 = float(np.max(ods) - baseline)
        t0_0 = float(times[np.argmin(np.abs(ods - (baseline + K0/2)))])
        def logistic_fixed(t, K, r, t0):
            return baseline + K / (1.0 + np.exp(-r * (t - t0)))
        try:
            popt, _ = curve_fit(logistic_fixed, times, ods, p0=[K0, 1.0, t0_0],
                               bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]),
                               maxfev=10000)
            K, r, t0 = popt
            y_pred = logistic_fixed(times, *popt)
            ss_res = np.sum((ods - y_pred)**2)
            ss_tot = np.sum((ods - np.mean(ods))**2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else None
            full_popt = np.array([K, r, t0, baseline])
            fit_results[well] = {
                'times': times, 'ods': ods, 'popt': full_popt,
                'K': K, 'r': r, 't0': t0, 'y0': baseline, 'r2': r2,
                'baseline': baseline, 'biomass_gain': K,
            }
        except Exception as e:
            fit_results[well] = {
                'times': times, 'ods': ods, 'popt': None,
                'K': None, 'r': None, 't0': None, 'y0': baseline, 'r2': None,
                'baseline': baseline, 'biomass_gain': None, 'error': str(e),
            }
    return fit_results

# Parse all rounds
r1_wells = parse_data(r1_csv_text, r1_conditions)
r2_wells = parse_data(r2_csv_text, r2_conditions)
r3_wells = parse_data(r3_csv_text, r3_conditions)

# Approach 1: Fitted offset (4 params)
r1_fits_fitted = fit_all_fitted_offset(r1_wells)
r2_fits_fitted = fit_all_fitted_offset(r2_wells)
r3_fits_fitted = fit_all_fitted_offset(r3_wells)

# Approach 2: Fixed offset (3 params)
r1_fits_fixed = fit_all_fixed_offset(r1_wells)
r2_fits_fixed = fit_all_fixed_offset(r2_wells)
r3_fits_fixed = fit_all_fixed_offset(r3_wells)

print("Approach 1 — Fitted offset (4 params: K, r, t0, y0)")
print(f"  Round 1: {len(r1_fits_fitted)} wells, Round 2: {len(r2_fits_fitted)} wells, Round 3: {len(r3_fits_fitted)} wells")
print("\nApproach 2 — Fixed offset (3 params: K, r, t0; y0 = first OD)")
print(f"  Round 1: {len(r1_fits_fixed)} wells, Round 2: {len(r2_fits_fixed)} wells, Round 3: {len(r3_fits_fixed)} wells")
print(f"\nNote: Round 3 has only {len(list(r3_wells.values())[0])} timepoints — fits are preliminary")

In [ ]:
def plot_round(fit_results, conditions, round_name, color_map, approach_name):
    sorted_wells = sorted(fit_results.keys(), key=lambda w: -(fit_results[w]['r'] or 0))
    n = len(sorted_wells)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4.2))
    axes = axes.flatten()
    for idx, well in enumerate(sorted_wells):
        ax = axes[idx]
        res = fit_results[well]
        cond_name, cond_detail = conditions[well]
        times = res['times']
        ods = res['ods']
        color = '#9E9E9E'
        for key, c in color_map.items():
            if key in cond_name:
                color = c
                break
        ax.scatter(times, ods, color=color, s=40, zorder=5)
        if res['popt'] is not None:
            t_smooth = np.linspace(0, max(times) * 1.1, 200)
            y_smooth = logistic(t_smooth, *res['popt'])
            ax.plot(t_smooth, y_smooth, color=color, linewidth=2, alpha=0.8)
            ax.axhline(y=res['y0'] + res['K'], color=color, linestyle='--', alpha=0.3)
            ax.axhline(y=res['y0'], color='gray', linestyle=':', alpha=0.3)
            ax.text(0.97, 0.03,
                    f"r = {res['r']:.3f} /hr\nR² = {res['r2']:.4f}\nK = {res['K']:.3f} (gain)\ny₀ = {res['y0']:.3f}",
                    transform=ax.transAxes, fontsize=9, va='bottom', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
        ax.set_title(f'{well}: {cond_name}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('OD600')
        ax.set_ylim(0, max(0.8, max(ods) * 1.15))
        ax.grid(True, alpha=0.3)
        ax.text(0.5, -0.15, cond_detail, transform=ax.transAxes, fontsize=8, ha='center', color='gray')
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    plt.suptitle(f'{round_name} — {approach_name} (sorted by growth rate)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    return fig

r1_colors = {'SD': '#4CAF50', 'HB': '#2196F3', 'LB': '#FF5722', 'NB': '#9E9E9E', 'CTL': '#9E9E9E'}
r2_colors = {'BayBE': '#2196F3', 'SD': '#4CAF50', 'Elnora': '#FF9800', 'CTL': '#9E9E9E', 'NB': '#9E9E9E'}
r3_colors = {'BayBE': '#2196F3', 'SD': '#4CAF50', 'Elnora': '#FF9800', 'CTL': '#9E9E9E', 'NB': '#9E9E9E'}

# === APPROACH 1: Fitted offset ===
print("=" * 60)
print("APPROACH 1: Fitted offset (4 params)")
print("=" * 60)
for name, fits, conds, colors in [
    ('Round 1', r1_fits_fitted, r1_conditions, r1_colors),
    ('Round 2', r2_fits_fitted, r2_conditions, r2_colors),
    ('Round 3 (preliminary)', r3_fits_fitted, r3_conditions, r3_colors),
]:
    fig = plot_round(fits, conds, name, colors, 'Fitted Offset')
    fig.savefig(f'{name.split()[0].lower()}{name.split()[1] if len(name.split())>1 else ""}_fitted_offset.png'.replace(' ','').replace('(','').replace(')',''), dpi=150, bbox_inches='tight')
    plt.show()

# === APPROACH 2: Fixed offset ===
print("=" * 60)
print("APPROACH 2: Fixed offset (3 params, y0 = first data point)")
print("=" * 60)
for name, fits, conds, colors in [
    ('Round 1', r1_fits_fixed, r1_conditions, r1_colors),
    ('Round 2', r2_fits_fixed, r2_conditions, r2_colors),
    ('Round 3 (preliminary)', r3_fits_fixed, r3_conditions, r3_colors),
]:
    fig = plot_round(fits, conds, name, colors, 'Fixed Offset (y0 = first OD)')
    fig.savefig(f'{name.split()[0].lower()}{name.split()[1] if len(name.split())>1 else ""}_fixed_offset.png'.replace(' ','').replace('(','').replace(')',''), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Combined scatter: both approaches stacked, all 3 rounds
from matplotlib.lines import Line2D

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 16))

for ax, approach, fits_list, title in [
    (ax1, 'Fitted', [(r1_fits_fitted, r1_conditions), (r2_fits_fitted, r2_conditions), (r3_fits_fitted, r3_conditions)], 'Fitted Offset (4 params)'),
    (ax2, 'Fixed', [(r1_fits_fixed, r1_conditions), (r2_fits_fixed, r2_conditions), (r3_fits_fixed, r3_conditions)], 'Fixed Offset (3 params)'),
]:
    markers = ['o', 's', 'D']
    labels = ['R1', 'R2', 'R3']
    for (fits, conditions), marker, rnd in zip(fits_list, markers, labels):
        for well in fits:
            res = fits[well]
            cond_name, _ = conditions[well]
            if res['r'] is None or res['biomass_gain'] is None or res['biomass_gain'] > 2:
                continue
            if res['r2'] is not None and res['r2'] < 0:
                continue
            if 'CTL' in cond_name or 'NB' in cond_name:
                color = '#9E9E9E'
            elif 'Elnora' in cond_name:
                color = '#FF9800'
            elif 'LB' in cond_name:
                color = '#FF5722'
            elif 'SD' in cond_name:
                color = '#4CAF50'
            else:
                color = '#2196F3'
            ax.scatter(res['r'], res['biomass_gain'], color=color, s=80, marker=marker, zorder=5, edgecolors='white', linewidth=0.5)
            ax.annotate(f"{rnd} {well}", (res['r'], res['biomass_gain']), fontsize=5, ha='center', va='bottom', xytext=(0, 6), textcoords='offset points')
    ax.set_xlabel('Growth Rate (per hour)')
    ax.set_ylabel('Biomass Gain (K)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2196F3', markersize=8, label='R1 HB'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4CAF50', markersize=8, label='R1 SD'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF5722', markersize=8, label='R1 LBv2'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#2196F3', markersize=8, label='R2 BayBE'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#FF9800', markersize=8, label='R2 Elnora'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#2196F3', markersize=8, label='R3 BayBE'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#FF9800', markersize=8, label='R3 Elnora'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#9E9E9E', markersize=8, label='Control'),
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=8)
plt.suptitle('Growth Rate vs Biomass Gain — All Rounds', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('all_rounds_rate_vs_biomass.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Combined summary table — all 3 rounds, both approaches
import pandas as pd

rows = []
for approach, all_fits in [
    ('Fitted Offset', [(r1_fits_fitted, r1_conditions, 'R1'), (r2_fits_fitted, r2_conditions, 'R2'), (r3_fits_fitted, r3_conditions, 'R3')]),
    ('Fixed Offset', [(r1_fits_fixed, r1_conditions, 'R1'), (r2_fits_fixed, r2_conditions, 'R2'), (r3_fits_fixed, r3_conditions, 'R3')]),
]:
    for fits, conditions, rnd in all_fits:
        for well in sorted(fits.keys(), key=lambda w: -(fits[w]['r'] or 0)):
            res = fits[well]
            cond_name, _ = conditions[well]
            rows.append({
                'Approach': approach,
                'Round': rnd,
                'Well': well,
                'Condition': cond_name,
                'r (/hr)': round(res['r'], 4) if res['r'] else None,
                'K (gain)': round(res['K'], 4) if res['K'] else None,
                'y0': round(res['y0'], 4) if res['y0'] else None,
                'Max OD': round(res['y0'] + res['K'], 4) if res['y0'] and res['K'] else None,
                'R²': round(res['r2'], 4) if res['r2'] else None,
            })

df = pd.DataFrame(rows)

# Show per round
for rnd in ['R1', 'R2', 'R3']:
    print(f"\n{'='*60}")
    print(f"  {rnd}" + (" (preliminary)" if rnd == 'R3' else ""))
    print(f"{'='*60}")
    sub = df[df['Round'] == rnd].pivot_table(
        index=['Well', 'Condition'],
        columns='Approach',
        values=['r (/hr)', 'K (gain)', 'R²'],
        aggfunc='first'
    )
    display(sub.sort_values(('r (/hr)', 'Fitted Offset'), ascending=False))